# FieldPack AI — Remote Ollama GPU Server (Secured)

Runs Gemma 4 E2B (Q4_K_M) on Colab's free T4 GPU and exposes it via
authenticated tunnel. All requests require a secret token.

**Usage:**
1. Set runtime to **GPU** (Runtime → Change runtime type → T4 GPU)
2. Run all cells
3. Copy the tunnel URL and secret token printed at the end
4. On your local machine, set in `.env`:
   ```
   OLLAMA_BASE_URL=<tunnel_url>
   OLLAMA_TUNNEL_TOKEN=<secret_token>
   ```
5. Run your local FastAPI app as normal — it talks to Colab's GPU

**Security:** A reverse proxy validates every request against a 64-char
random token via `Authorization: Bearer <token>` header. No token = 403.

In [ ]:
# Cell 1: Verify GPU
!nvidia-smi
import torch
assert torch.cuda.is_available(), "No GPU detected! Change runtime to T4 GPU."
gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"\nGPU: {gpu_name} ({gpu_mem:.1f} GB) -- Ready")

In [ ]:
# Cell 2: Install Ollama
!apt-get update -qq && apt-get install -y -qq zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
!ollama --version

In [ ]:
# Cell 3: Start Ollama server in background
import subprocess, time, requests

proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout=open("/tmp/ollama.log", "w"),
    stderr=subprocess.STDOUT,
)
print(f"Ollama server PID: {proc.pid}")

for i in range(30):
    try:
        r = requests.get("http://localhost:11434/api/version", timeout=2)
        if r.status_code == 200:
            print(f"Ollama ready: {r.json()}")
            break
    except:
        pass
    time.sleep(1)
else:
    raise RuntimeError("Ollama failed to start. Check /tmp/ollama.log")

In [ ]:
# Cell 4: Pull Gemma 4 E2B (Q4_K_M) -- ~4.5 GB download, 2-5 min
!ollama pull gemma4:e2b-it-q4_K_M

In [ ]:
# Cell 5: Warm up + benchmark
import requests, time

print("Warming up model (first run compiles CUDA kernels)...")
start = time.time()
requests.post("http://localhost:11434/api/generate", json={
    "model": "gemma4:e2b-it-q4_K_M",
    "prompt": "Hello",
    "stream": False,
    "options": {"num_predict": 10}
})
print(f"Warm-up done in {time.time() - start:.1f}s")

print("\nBenchmarking (256 tokens)...")
r = requests.post("http://localhost:11434/api/generate", json={
    "model": "gemma4:e2b-it-q4_K_M",
    "prompt": "Explain cassava mosaic disease in 200 words.",
    "stream": False,
    "options": {"num_predict": 256}
})
d = r.json()
eval_dur = d.get("eval_duration", 0) / 1e9
eval_count = d.get("eval_count", 0)
tok_s = eval_count / eval_dur if eval_dur > 0 else 0
print(f"Generated {eval_count} tokens at {tok_s:.1f} tok/s")

ps = requests.get("http://localhost:11434/api/ps").json()
for m in ps.get("models", []):
    vram_gb = m.get("size_vram", 0) / 1e9
    total_gb = m.get("size", 0) / 1e9
    print(f"Model VRAM: {vram_gb:.1f}GB / {total_gb:.1f}GB on GPU")

In [ ]:
# Cell 6: Auth proxy + tunnel
#
# Generates a 64-char random token. Runs a threaded HTTP proxy on
# port 11435 that checks Authorization header before forwarding to
# Ollama on 11434. The tunnel exposes 11435, NOT 11434 directly.

import json, secrets, subprocess, re, time, threading
from http.server import ThreadingHTTPServer, BaseHTTPRequestHandler
import requests as upstream_requests

# Generate a random 64-character hex token
SECRET_TOKEN = secrets.token_hex(32)

OLLAMA_UPSTREAM = "http://localhost:11434"
PROXY_PORT = 11435


class AuthProxy(BaseHTTPRequestHandler):
    """Reverse proxy that validates Bearer token before forwarding."""

    def do_request(self):
        auth = self.headers.get("Authorization", "")
        if auth != f"Bearer {SECRET_TOKEN}":
            self.send_response(403)
            self.send_header("Content-Type", "application/json")
            self.end_headers()
            self.wfile.write(b'{"error": "forbidden"}')
            return

        # Read request body
        content_len = int(self.headers.get("Content-Length", 0))
        body = self.rfile.read(content_len) if content_len > 0 else None

        # Forward to Ollama (strip auth/host headers)
        url = f"{OLLAMA_UPSTREAM}{self.path}"
        headers = {k: v for k, v in self.headers.items()
                   if k.lower() not in ("host", "authorization")}

        try:
            resp = upstream_requests.request(
                self.command, url, headers=headers, data=body,
                stream=True, timeout=(10, 300),
            )
            self.send_response(resp.status_code)
            for key, val in resp.headers.items():
                if key.lower() not in ("transfer-encoding", "content-length"):
                    self.send_header(key, val)
            self.send_header("Connection", "close")
            self.end_headers()
            for line in resp.iter_lines():
                if line:
                    self.wfile.write(line + b"\n")
                    self.wfile.flush()
        except Exception as e:
            self.send_response(502)
            self.send_header("Content-Type", "application/json")
            self.end_headers()
            self.wfile.write(json.dumps({"error": str(e)}).encode())

    # Handle all HTTP methods
    do_GET = do_request
    do_POST = do_request
    do_PUT = do_request
    do_DELETE = do_request
    do_HEAD = do_request

    def log_message(self, format, *args):
        pass  # Silence request logs


# Start threaded proxy (handles concurrent requests)
server = ThreadingHTTPServer(("127.0.0.1", PROXY_PORT), AuthProxy)
proxy_thread = threading.Thread(target=server.serve_forever, daemon=True)
proxy_thread.start()
print(f"Auth proxy running on port {PROXY_PORT} (threaded)")

# Verify proxy rejects unauthenticated requests
import requests as req
r = req.get(f"http://localhost:{PROXY_PORT}/api/version")
assert r.status_code == 403, f"Proxy should reject! Got {r.status_code}"
r = req.get(f"http://localhost:{PROXY_PORT}/api/version",
            headers={"Authorization": f"Bearer {SECRET_TOKEN}"})
assert r.status_code == 200, f"Proxy should allow! Got {r.status_code}"
print("Auth proxy verified: rejects without token, allows with token")

# Install and start cloudflared tunnel (points to PROXY, not Ollama)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared 2>/dev/null
!chmod +x /usr/local/bin/cloudflared

tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", f"http://localhost:{PROXY_PORT}"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

tunnel_url = None
for i in range(30):
    line = tunnel.stdout.readline()
    if line:
        match = re.search(r'(https://[\w-]+\.trycloudflare\.com)', line)
        if match:
            tunnel_url = match.group(1)
            break
    time.sleep(0.5)

if tunnel_url:
    print()
    print("=" * 60)
    print("  CONNECTION DETAILS (copy to your local .env)")
    print("=" * 60)
    print(f"  OLLAMA_BASE_URL={tunnel_url}")
    print(f"  OLLAMA_TUNNEL_TOKEN={SECRET_TOKEN}")
    print("=" * 60)
    print()
    print("Without the token, all requests get 403 Forbidden.")
    print("Token is 64-char random hex, regenerated every session.")
else:
    print("ERROR: Failed to get tunnel URL.")

In [ ]:
# Cell 7: Keep alive -- prevents Colab from disconnecting
# Run this, then switch to your local machine. Stop manually when done.
import time, requests

print("Keeping server alive. Press Stop when done recording.")
print(f"Tunnel: {tunnel_url}")
print()

fails = 0
while True:
    try:
        r = requests.get(f"http://localhost:{PROXY_PORT}/api/version",
                         headers={"Authorization": f"Bearer {SECRET_TOKEN}"},
                         timeout=5)
        status = "OK" if r.status_code == 200 else f"ERR {r.status_code}"
        fails = 0
    except Exception as e:
        fails += 1
        status = f"ERR ({fails}x): {e}"
        if fails >= 5:
            print(f"\n[{time.strftime('%H:%M:%S')}] Ollama appears down. Restarting...")
            subprocess.Popen(["ollama", "serve"],
                             stdout=open("/tmp/ollama.log", "a"),
                             stderr=subprocess.STDOUT)
            time.sleep(10)
            fails = 0
    print(f"[{time.strftime('%H:%M:%S')}] Ollama: {status}       ", end="\r")
    time.sleep(30)